# 01. Snowflake: Setup and the Semantic View (Demo Flow)

One semantic model, two platforms, moved by hand so you can see each step. Nobody
re-implements anybody else's metric.

```
        SNOWFLAKE                    S3 (your bucket)                 DATABRICKS
  +-------------------+                                          +-------------------+
  |   SALES_SV        |                                          | sales_metric_view |
  |  (Semantic View)  |                                          |   (Metric View)   |
  +---------+---------+          ossie/demo/*.yaml               +---------+---------+
            |                                                             |
            +--- export -----------> [ Ossie ] -----------> import -------+
            +--- import <----------- [  YAML ] <----------- export -------+
            |                                                             |
  +---------+---------+          iceberg/customers.../            +-------+---------+
  | CUSTOMERS, ORDERS |  ------>  iceberg/orders.../     <------- | customers,      |
  | (Iceberg tables)  |          (Parquet + metadata)             | orders          |
  +-------------------+                                           +-----------------+
                            same physical files, no copy
```

Two things are shared: the **data**, through Iceberg on S3, and the **meaning**, through
an Apache Ossie file on the same bucket.

Objects live in two schemas, and the split matters:

| Schema | Holds | Why |
|---|---|---|
| `DEMO_SEMANTIC_INTEROP` | semantic view, stage, tasks | one per flow, so the demo, live and snowflake_managed variants coexist |
| `EXT_SEMANTIC_INTEROP` | Iceberg tables, external volume | shared, because Snowflake gives each Iceberg table a random S3 suffix and Databricks discovers tables by scanning for those |

You will visit four notebooks in this order:

| Order | Notebook | Platform | What happens |
|---|---|---|---|
| 1 | `Snowflake/01_setup` (this one) | Snowflake | Verify the data, create the semantic view |
| 2 | `Snowflake/02_demo_and_export` | Snowflake | Show the data and the model, export to Ossie |
| 3 | `DBX/01_ossie_to_metric_view` | Databricks | Build the metric view, add a measure, export back |
| 4 | `Snowflake/03_import_from_ossie` | Snowflake | Import, and the Databricks measure appears |

Prerequisites: `setup/snowflake_setup.sql` has been run once for the account, and the
AWS IAM trust policy allows the storage integration and external volume. See
`setup/COCO_SETUP_GUIDE.md`.


## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "DEMO_SEMANTIC_INTEROP"   # semantic view, stage and tasks
DATA_SCHEMA = "EXT_SEMANTIC_INTEROP"   # Iceberg tables, shared with the other flows
SEMANTIC_VIEW_NAME  = 'SALES_SV'
STAGE_NAME = 'DEMO_OSSIE_STAGE'
print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Verify Iceberg tables on S3

The tables are Snowflake-managed Iceberg tables stored on S3. Both Snowflake and
Databricks read from the same physical Parquet files.

In [ ]:
%%sql -r dataframe_2
-- Expected: EAST 750/5/12, WEST 700/5/11
SELECT 
    c.region, 
    SUM(o.order_amount) AS total_amount, 
    COUNT(o.order_id) AS order_count, 
    SUM(o.order_qty) AS total_qty
FROM {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS o
JOIN {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS c USING (customer_id)
GROUP BY c.region ORDER BY c.region;

In [ ]:
%%sql -r dataframe_5
-- Enable directory table on the stage (one-time)
ALTER STAGE {{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}
  SET DIRECTORY = (ENABLE = TRUE);


## Step 3 - Create the semantic view

Two metrics (`TOTAL_ORDER_AMOUNT`, `ORDER_COUNT`), two dimensions, and the
ORDERS-to-CUSTOMERS relationship.

Every dimension and metric carries synonyms. They are the business vocabulary that lets
Cortex Analyst resolve "revenue" or "units" to the right metric, and they are worth
watching during the demo: they travel with the model into the Databricks Metric View,
where they appear in the `synonyms` field and can be edited.


In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.SALES_SV
  TABLES (
    orders AS {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS PRIMARY KEY (order_id),
    customers AS {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region
      WITH SYNONYMS ('area', 'territory')
      COMMENT = 'Sales region',
    customers.customer_name AS customer_name
      WITH SYNONYMS ('client', 'account')
      COMMENT = 'Customer name'
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount)
      WITH SYNONYMS ('revenue', 'sales')
      COMMENT = 'Total order amount',
    orders.order_count AS COUNT(orders.order_id)
      WITH SYNONYMS ('orders', 'transactions')
      COMMENT = 'Number of orders'
  )
  COMMENT = 'Sales star for Ossie interop demo (Iceberg on S3)';

In [ ]:
%%sql -r dataframe_4
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.SALES_SV
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;